In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score

In [2]:

# Step 2: Load the raw dataset

from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

fixed_path = PROJECT_DIR / "data" / "raw" / "fixed-text.csv"

df_fixed = pd.read_csv(fixed_path)

print("Dataset shape:", df_fixed.shape)
print("Column names:")
print(df_fixed.columns.tolist())

print("\nFirst five rows:")
display(df_fixed.head())


Dataset shape: (19772, 50)
Column names:
['participant', 'session', 'repetition', 'DU.v.v', 'DD.v.p', 'DU.v.p', 'UD.v.p', 'UU.v.p', 'DU.p.p', 'DD.p.w', 'DU.p.w', 'UD.p.w', 'UU.p.w', 'DU.w.w', 'DD.w.j', 'DU.w.j', 'UD.w.j', 'UU.w.j', 'DU.j.j', 'DD.j.k', 'DU.j.k', 'UD.j.k', 'UU.j.k', 'DU.k.k', 'DD.k.e', 'DU.k.e', 'UD.k.e', 'UU.k.e', 'DU.e.e', 'DD.e.u', 'DU.e.u', 'UD.e.u', 'UU.e.u', 'DU.u.u', 'DD.u.r', 'DU.u.r', 'UD.u.r', 'UU.u.r', 'DU.r.r', 'DD.r.k', 'DU.r.k', 'UD.r.k', 'UU.r.k', 'DU.k.k.1', 'DD.k.b', 'DU.k.b', 'UD.k.b', 'UU.k.b', 'DU.b.b', 'total time']

First five rows:


,participant,session,repetition,DU.v.v,DD.v.p,DU.v.p,UD.v.p,UU.v.p,DU.p.p,DD.p.w,...,DU.r.k,UD.r.k,UU.r.k,DU.k.k.1,DD.k.b,DU.k.b,UD.k.b,UU.k.b,DU.b.b,total time
0,p001,1,1,0.129,1.917,1.804,2.046,1.933,0.113,0.719,...,1.289,1.509,1.423,0.086,0.450,0.352,0.536,0.438,0.098,10.101
1,p001,1,2,0.112,0.192,0.096,0.304,0.208,0.096,0.448,...,0.921,1.135,1.056,0.079,0.418,0.336,0.497,0.415,0.082,5.768
2,p001,1,3,0.088,0.253,0.182,0.341,0.270,0.071,1.536,...,1.066,1.271,1.194,0.077,0.448,0.352,0.525,0.429,0.096,7.877
3,p001,1,4,0.104,0.323,0.233,0.427,0.337,0.090,0.912,...,0.927,1.129,1.063,0.066,0.418,0.330,0.484,0.396,0.088,5.443
4,p001,1,5,0.107,0.188,0.100,0.295,0.207,0.088,0.858,...,1.237,1.423,1.344,0.079,0.457,0.369,0.536,0.448,0.088,5.284


In [3]:

# Step 3: Inspect dataset structure

print("Available columns:")
print(df_fixed.columns.tolist())

print("\nMissing values:", df_fixed.isna().sum().sum())

print("\nSession distribution:")
print(df_fixed["session"].value_counts().sort_index())

print("\nParticipant count:", df_fixed["participant"].nunique())


Available columns:
['participant', 'session', 'repetition', 'DU.v.v', 'DD.v.p', 'DU.v.p', 'UD.v.p', 'UU.v.p', 'DU.p.p', 'DD.p.w', 'DU.p.w', 'UD.p.w', 'UU.p.w', 'DU.w.w', 'DD.w.j', 'DU.w.j', 'UD.w.j', 'UU.w.j', 'DU.j.j', 'DD.j.k', 'DU.j.k', 'UD.j.k', 'UU.j.k', 'DU.k.k', 'DD.k.e', 'DU.k.e', 'UD.k.e', 'UU.k.e', 'DU.e.e', 'DD.e.u', 'DU.e.u', 'UD.e.u', 'UU.e.u', 'DU.u.u', 'DD.u.r', 'DU.u.r', 'UD.u.r', 'UU.u.r', 'DU.r.r', 'DD.r.k', 'DU.r.k', 'UD.r.k', 'UU.r.k', 'DU.k.k.1', 'DD.k.b', 'DU.k.b', 'UD.k.b', 'UU.k.b', 'DU.b.b', 'total time']

Missing values: 0

Session distribution:
session
1    9878
2    9894
Name: count, dtype: int64

Participant count: 99


In [4]:
# Step 4: Reproduce the original modeling-data preparation

# Start from the raw dataset so this cell is independent of prior notebook state.
df_model = df_fixed.copy()

excluded_records = [
    ("p020", 2, 4),
    ("p071", 1, 20),
    ("p087", 1, 8),
]

for participant, session, repetition in excluded_records:
    record_mask = (
        df_model["participant"].eq(participant)
        & df_model["session"].eq(session)
        & df_model["repetition"].eq(repetition)
    )
    matching_records = int(record_mask.sum())
    if matching_records != 1:
        raise ValueError(
            f"Expected exactly one excluded record for "
            f"{(participant, session, repetition)}, found {matching_records}."
        )
    df_model = df_model.loc[~record_mask].copy()

df_model.reset_index(drop=True, inplace=True)

print("Raw records:", len(df_fixed))
print("Excluded records:", len(excluded_records))
print("Modeling records:", len(df_model))

# Match the original notebook's feature-selection rule and column order.
excluded_columns = [
    "participant",
    "session",
    "repetition",
    "total time"
]

feature_columns = [
    col for col in df_model.columns
    if col not in excluded_columns
]

X = df_model[feature_columns].apply(
    pd.to_numeric, errors="raise"
)
y = df_model["participant"].copy()

if len(feature_columns) != 46:
    raise ValueError(f"Expected 46 timing features, found {len(feature_columns)}.")

print("Feature matrix shape:", X.shape)
print("Number of features:", X.shape[1])
print("Number of participants:", y.nunique())
print("'total time' excluded:", "total time" not in X.columns)

Raw records: 19772
Excluded records: 3
Modeling records: 19769
Feature matrix shape: (19769, 46)
Number of features: 46
Number of participants: 99
'total time' excluded: True


In [5]:
# Step 5: Encode participant labels and split by session

from sklearn.preprocessing import LabelEncoder

train_mask = df_model["session"].eq(1)
test_mask = df_model["session"].eq(2)
if not (train_mask | test_mask).all():
    raise ValueError("Modeling data contains sessions other than 1 and 2.")

X_train = X.loc[train_mask].copy()
X_test = X.loc[test_mask].copy()
y_train = y.loc[train_mask].copy()
y_test = y.loc[test_mask].copy()

label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)

unseen_test_participants = set(y_test.unique()) - set(label_encoder.classes_)
if unseen_test_participants:
    raise ValueError(
        f"Test participants missing from training classes: "
        f"{sorted(unseen_test_participants)}"
    )
y_test_encoded = label_encoder.transform(y_test)

if len(X_train) != 9876 or len(X_test) != 9893:
    raise ValueError(
        f"Unexpected session counts: train={len(X_train)}, test={len(X_test)}."
    )
if not X_train.index.equals(y_train.index) or not X_test.index.equals(y_test.index):
    raise ValueError("Feature rows and participant labels are misaligned.")
if not np.array_equal(label_encoder.inverse_transform(y_train_encoded), y_train.to_numpy()):
    raise ValueError("Training labels do not match the fitted label encoding.")
if not np.array_equal(label_encoder.inverse_transform(y_test_encoded), y_test.to_numpy()):
    raise ValueError("Testing labels do not match the training label encoding.")

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)
print("Training participants:", y_train.nunique())
print("Testing participants:", y_test.nunique())
print("Encoded label range:", int(y_train_encoded.min()), "to", int(y_train_encoded.max()))

Training shape: (9876, 46)
Testing shape: (9893, 46)
Training participants: 99
Testing participants: 99
Encoded label range: 0 to 98


In [6]:
# Step 6: Scale using training-session statistics only

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Preprocessing integrity checks; these do not fit on test-session data.
feature_order_matches_original_rule = feature_columns == [
    col for col in df_model.columns
    if col not in excluded_columns
]
feature_alignment = (
    feature_order_matches_original_rule
    and list(X.columns) == feature_columns
    and list(X_train.columns) == feature_columns
    and list(X_test.columns) == feature_columns
)
missing_train = int(X_train.isna().sum().sum())
missing_test = int(X_test.isna().sum().sum())
infinite_train = int(np.isinf(X_train.to_numpy(dtype=float)).sum())
infinite_test = int(np.isinf(X_test.to_numpy(dtype=float)).sum())
scaled_missing = int(np.isnan(X_train_scaled).sum() + np.isnan(X_test_scaled).sum())
scaled_infinite = int(np.isinf(X_train_scaled).sum() + np.isinf(X_test_scaled).sum())

if not feature_alignment:
    raise ValueError("Feature names or column order are not aligned.")
if missing_train or missing_test or infinite_train or infinite_test:
    raise ValueError("Input features contain missing or infinite values.")
if scaled_missing or scaled_infinite:
    raise ValueError("Scaled features contain missing or infinite values.")
if len(label_encoder.classes_) != 99:
    raise ValueError(f"Expected 99 participant classes, found {len(label_encoder.classes_)}.")
if scaler.n_samples_seen_ != len(X_train):
    raise ValueError("Scaler was not fitted using exactly the training rows.")
np.testing.assert_allclose(
    scaler.mean_, X_train.mean(axis=0).to_numpy(dtype=float), rtol=0, atol=1e-12
)
np.testing.assert_allclose(
    X_test_scaled, scaler.transform(X_test), rtol=0, atol=0
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))
print("Feature count:", len(feature_columns))
print("Training/testing participant counts:", y_train.nunique(), y_test.nunique())
print("Training/testing encoded label ranges:",
      (int(y_train_encoded.min()), int(y_train_encoded.max())),
      (int(y_test_encoded.min()), int(y_test_encoded.max())))
print("Feature-column alignment:", feature_alignment)
print("Missing values (train/test):", missing_train, missing_test)
print("Infinite values (train/test):", infinite_train, infinite_test)
print("Missing/infinite values after scaling:", scaled_missing, scaled_infinite)
print("Test participants covered by training encoder:",
      set(y_test.unique()).issubset(set(label_encoder.classes_)))
print("Encoder inverse-transform alignment (train/test):",
      np.array_equal(label_encoder.inverse_transform(y_train_encoded), y_train.to_numpy()),
      np.array_equal(label_encoder.inverse_transform(y_test_encoded), y_test.to_numpy()))
print("Scaler fitted on training rows only:", scaler.n_samples_seen_ == len(X_train))
print("Scaler mean matches training features:",
      np.allclose(scaler.mean_, X_train.mean(axis=0).to_numpy(dtype=float), rtol=0, atol=1e-12))

Training samples: 9876
Testing samples: 9893
Feature count: 46
Training/testing participant counts: 99 99
Training/testing encoded label ranges: (0, 98) (0, 98)
Feature-column alignment: True
Missing values (train/test): 0 0
Infinite values (train/test): 0 0
Missing/infinite values after scaling: 0 0
Test participants covered by training encoder: True
Encoder inverse-transform alignment (train/test): True True
Scaler fitted on training rows only: True
Scaler mean matches training features: True


In [11]:

# Step 7: Check the feature columns

print("Number of features:", len(feature_columns))

print("\nAll feature columns:")
for i, col in enumerate(feature_columns, start=1):
    print(f"{i}. {col}")

print("\nCheck whether 'total time' is included:")
print("total time" in feature_columns)



Number of features: 46

All feature columns:
1. DU.v.v
2. DD.v.p
3. DU.v.p
4. UD.v.p
5. UU.v.p
6. DU.p.p
7. DD.p.w
8. DU.p.w
9. UD.p.w
10. UU.p.w
11. DU.w.w
12. DD.w.j
13. DU.w.j
14. UD.w.j
15. UU.w.j
16. DU.j.j
17. DD.j.k
18. DU.j.k
19. UD.j.k
20. UU.j.k
21. DU.k.k
22. DD.k.e
23. DU.k.e
24. UD.k.e
25. UU.k.e
26. DU.e.e
27. DD.e.u
28. DU.e.u
29. UD.e.u
30. UU.e.u
31. DU.u.u
32. DD.u.r
33. DU.u.r
34. UD.u.r
35. UU.u.r
36. DU.r.r
37. DD.r.k
38. DU.r.k
39. UD.r.k
40. UU.r.k
41. DU.k.k.1
42. DD.k.b
43. DU.k.b
44. UD.k.b
45. UU.k.b
46. DU.b.b

Check whether 'total time' is included:
False


In [7]:
# Step 10: Train the original MLP configuration on a Session 1 fit split

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.neural_network import MLPClassifier

VALIDATION_SIZE = 0.20
SPLIT_RANDOM_STATE = 42
MODEL_RANDOM_STATE = 42

# Hold out 20% of Session 1, stratified so all participant classes are represented.
X_fit, X_val, y_fit, y_val = train_test_split(
    X_train,
    y_train,
    test_size=VALIDATION_SIZE,
    random_state=SPLIT_RANDOM_STATE,
    stratify=y_train,
)

# Refit both learned preprocessing steps using only this fit split.
baseline_label_encoder = LabelEncoder()
y_fit_encoded = baseline_label_encoder.fit_transform(y_fit)
if not (y_val.isin(baseline_label_encoder.classes_)).all():
    raise ValueError("Validation contains a participant absent from the fit split.")
y_val_encoded = baseline_label_encoder.transform(y_val)
if not np.array_equal(baseline_label_encoder.classes_, label_encoder.classes_):
    raise ValueError("Fit-split label mapping differs from the established class mapping.")

baseline_scaler = StandardScaler()
X_fit_scaled = baseline_scaler.fit_transform(X_fit)
X_val_scaled = baseline_scaler.transform(X_val)

# Configuration matches the original notebook's improved MLP baseline.
baseline_config = {
    "hidden_layer_sizes": (128, 64),
    "activation": "relu",
    "solver": "adam",
    "batch_size": 256,
    "learning_rate_init": 0.0005,
    "alpha": 0.001,
    "max_iter": 150,
    "early_stopping": True,
    "validation_fraction": 0.15,
    "n_iter_no_change": 10,
    "tol": 0.0001,
    "random_state": MODEL_RANDOM_STATE,
}

mlp_baseline = MLPClassifier(**baseline_config, verbose=False)
mlp_baseline.fit(X_fit_scaled, y_fit_encoded)

print("Session 1 fit samples:", len(X_fit))
print("Session 1 validation samples:", len(X_val))
print("Session 2 held-out test samples:", len(X_test), "(not evaluated)")
print("Fit/validation participants:", y_fit.nunique(), y_val.nunique())
print("Iterations:", mlp_baseline.n_iter_)
print("Internal early-stopping validation accuracy:",
      f"{mlp_baseline.best_validation_score_:.4f}")
print("Configuration:", baseline_config)

Session 1 fit samples: 7900
Session 1 validation samples: 1976
Session 2 held-out test samples: 9893 (not evaluated)
Fit/validation participants: 99 99
Iterations: 111
Internal early-stopping validation accuracy: 0.6827
Configuration: {'hidden_layer_sizes': (128, 64), 'activation': 'relu', 'solver': 'adam', 'batch_size': 256, 'learning_rate_init': 0.0005, 'alpha': 0.001, 'max_iter': 150, 'early_stopping': True, 'validation_fraction': 0.15, 'n_iter_no_change': 10, 'tol': 0.0001, 'random_state': 42}


In [8]:
# Step 11: Evaluate fit and validation splits; keep Session 2 untouched

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
)

fit_predictions = mlp_baseline.predict(X_fit_scaled)
validation_predictions = mlp_baseline.predict(X_val_scaled)


def classification_metrics(y_true, y_pred):
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="macro",
        zero_division=0,
    )
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": f1,
    }


fit_metrics = classification_metrics(y_fit_encoded, fit_predictions)
validation_metrics = classification_metrics(y_val_encoded, validation_predictions)

baseline_result = {
    "experiment": "original_mlp_baseline",
    "hidden_layer_sizes": str(baseline_config["hidden_layer_sizes"]),
    "activation": baseline_config["activation"],
    "solver": baseline_config["solver"],
    "batch_size": baseline_config["batch_size"],
    "learning_rate_init": baseline_config["learning_rate_init"],
    "alpha": baseline_config["alpha"],
    "max_iter": baseline_config["max_iter"],
    "early_stopping": baseline_config["early_stopping"],
    "validation_fraction_internal": baseline_config["validation_fraction"],
    "random_state": MODEL_RANDOM_STATE,
    "split_random_state": SPLIT_RANDOM_STATE,
    "fit_samples": len(X_fit),
    "validation_samples": len(X_val),
    "fit_iterations": mlp_baseline.n_iter_,
    **{f"fit_{name}": value for name, value in fit_metrics.items()},
    **{f"validation_{name}": value for name, value in validation_metrics.items()},
    "session_2_evaluated": False,
}

mlp_validation_results = pd.DataFrame([baseline_result])
print("Fit-split metrics (macro precision/recall/F1):")
print({name: round(value, 4) for name, value in fit_metrics.items()})
print("Validation metrics (macro precision/recall/F1):")
print({name: round(value, 4) for name, value in validation_metrics.items()})
print("Structured experiment results:")
display(mlp_validation_results)

results_path = PROJECT_DIR / "results" / "metrics" / "mlp_validation_experiments.csv"
if results_path.exists():
    saved_results = pd.read_csv(results_path)
    baseline_rows = saved_results.loc[
        saved_results["experiment"] == baseline_result["experiment"]
    ]
    if len(baseline_rows) != 1:
        raise ValueError("Expected exactly one persisted baseline result row.")
    print("Existing baseline result retained; it was not overwritten.")
else:
    mlp_validation_results.to_csv(results_path, index=False)
    print("Saved validation results to:", results_path)

Fit-split metrics (macro precision/recall/F1):
{'accuracy': 0.8106, 'balanced_accuracy': 0.8107, 'macro_precision': 0.81, 'macro_recall': 0.8107, 'macro_f1': 0.8091}
Validation metrics (macro precision/recall/F1):
{'accuracy': 0.6675, 'balanced_accuracy': 0.6672, 'macro_precision': 0.6705, 'macro_recall': 0.6672, 'macro_f1': 0.6637}
Structured experiment results:


              experiment hidden_layer_sizes activation solver  batch_size  \
0  original_mlp_baseline          (128, 64)       relu   adam         256   

   learning_rate_init  alpha  max_iter  early_stopping  \
0              0.0005  0.001       150            True   

   validation_fraction_internal  ...  fit_balanced_accuracy  \
0                          0.15  ...               0.810652   

   fit_macro_precision  fit_macro_recall  fit_macro_f1  validation_accuracy  \
0             0.809957          0.810652      0.809074              0.66751   

   validation_balanced_accuracy  validation_macro_precision  \
0                      0.667225                    0.670497   

   validation_macro_recall  validation_macro_f1  session_2_evaluated  
0                 0.667225             0.663739                False  

[1 rows x 26 columns]

Existing baseline result retained; it was not overwritten.


In [9]:
# Step 12: Compare controlled MLP variants on the same Session 1 validation split

from sklearn.neural_network import MLPClassifier

results_path = PROJECT_DIR / "results" / "metrics" / "mlp_validation_experiments.csv"
if not results_path.exists():
    raise FileNotFoundError("Run and persist the baseline validation cell first.")
saved_results = pd.read_csv(results_path)

candidate_updates = [
    ("smaller_hidden_layers_64x32", {"hidden_layer_sizes": (64, 32)}),
    ("higher_alpha_0.01", {"alpha": 0.01}),
    ("higher_learning_rate_0.001", {"learning_rate_init": 0.001}),
    ("early_stopping_disabled", {"early_stopping": False}),
]
candidate_names = [name for name, _ in candidate_updates]
if saved_results["experiment"].duplicated().any():
    raise ValueError("Persisted validation results contain duplicate experiments.")
if not (saved_results["experiment"] == "original_mlp_baseline").sum() == 1:
    raise ValueError("Persisted results must contain exactly one original baseline.")

mlp_experiment_configs = {"original_mlp_baseline": dict(baseline_config)}
for experiment_name, parameter_update in candidate_updates:
    experiment_config = dict(baseline_config)
    experiment_config.update(parameter_update)
    mlp_experiment_configs[experiment_name] = experiment_config

saved_names = set(saved_results["experiment"])
existing_candidates = set(candidate_names) & saved_names
if existing_candidates and existing_candidates != set(candidate_names):
    raise ValueError("Only some candidate results are persisted; refusing partial reruns.")

if existing_candidates:
    mlp_validation_results = saved_results.copy()
    print("Loaded persisted candidate metrics; candidate models were not retrained.")
else:
    candidate_rows = []
    for experiment_name, _ in candidate_updates:
        experiment_config = mlp_experiment_configs[experiment_name]
        candidate_model = MLPClassifier(**experiment_config, verbose=False)
        candidate_model.fit(X_fit_scaled, y_fit_encoded)
        fit_predictions = candidate_model.predict(X_fit_scaled)
        validation_predictions = candidate_model.predict(X_val_scaled)
        candidate_row = {
            "experiment": experiment_name,
            "hidden_layer_sizes": str(experiment_config["hidden_layer_sizes"]),
            "activation": experiment_config["activation"],
            "solver": experiment_config["solver"],
            "batch_size": experiment_config["batch_size"],
            "learning_rate_init": experiment_config["learning_rate_init"],
            "alpha": experiment_config["alpha"],
            "max_iter": experiment_config["max_iter"],
            "early_stopping": experiment_config["early_stopping"],
            "validation_fraction_internal": experiment_config["validation_fraction"],
            "random_state": MODEL_RANDOM_STATE,
            "split_random_state": SPLIT_RANDOM_STATE,
            "fit_samples": len(X_fit),
            "validation_samples": len(X_val),
            "fit_iterations": candidate_model.n_iter_,
            **{
                f"fit_{name}": value
                for name, value in classification_metrics(y_fit_encoded, fit_predictions).items()
            },
            **{
                f"validation_{name}": value
                for name, value in classification_metrics(y_val_encoded, validation_predictions).items()
            },
            "session_2_evaluated": False,
        }
        candidate_rows.append(candidate_row)
        print(
            experiment_name,
            "iterations:", candidate_model.n_iter_,
            "validation macro-F1:", f"{candidate_row['validation_macro_f1']:.4f}",
            "validation balanced accuracy:",
            f"{candidate_row['validation_balanced_accuracy']:.4f}",
        )

    candidate_results = pd.DataFrame(candidate_rows)
    mlp_validation_results = pd.concat(
        [saved_results, candidate_results],
        ignore_index=True,
    )
    mlp_validation_results.to_csv(results_path, index=False)

ranked_validation_results = mlp_validation_results.sort_values(
    ["validation_macro_f1", "validation_balanced_accuracy", "validation_accuracy"],
    ascending=False,
).reset_index(drop=True)
print()
print("Validation results ranked by macro-F1, then balanced accuracy:")
display(
    ranked_validation_results[
        [
            "experiment",
            "hidden_layer_sizes",
            "learning_rate_init",
            "alpha",
            "early_stopping",
            "fit_accuracy",
            "validation_accuracy",
            "validation_balanced_accuracy",
            "validation_macro_precision",
            "validation_macro_recall",
            "validation_macro_f1",
        ]
    ]
)
print("Validation results:", results_path)

Loaded persisted candidate metrics; candidate models were not retrained.

Validation results ranked by macro-F1, then balanced accuracy:


                    experiment hidden_layer_sizes  learning_rate_init  alpha  \
0            higher_alpha_0.01          (128, 64)              0.0005  0.010   
1      early_stopping_disabled          (128, 64)              0.0005  0.001   
2        original_mlp_baseline          (128, 64)              0.0005  0.001   
3   higher_learning_rate_0.001          (128, 64)              0.0010  0.001   
4  smaller_hidden_layers_64x32           (64, 32)              0.0005  0.001   

   early_stopping  fit_accuracy  validation_accuracy  \
0            True      0.819241             0.678644   
1           False      0.897975             0.674089   
2            True      0.810633             0.667510   
3            True      0.790759             0.655364   
4            True      0.727089             0.629049   

   validation_balanced_accuracy  validation_macro_precision  \
0                      0.678416                    0.683960   
1                      0.673977                    0.677

Validation results: C:\Users\user\Desktop\DeepKey_DL_Project\results\metrics\mlp_validation_experiments.csv


In [10]:
# Step 13: Refit the validation-selected configuration and test Session 2 once

from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier

final_results_path = PROJECT_DIR / "results" / "metrics" / "mlp_cross_session_final.csv"
if final_results_path.exists():
    final_session2_results = pd.read_csv(final_results_path)
    print("Loaded the previously recorded one-time Session 2 evaluation; it was not rerun.")
    display(final_session2_results)
else:
    selected_result = ranked_validation_results.iloc[0]
    selected_experiment = selected_result["experiment"]
    selected_config = dict(mlp_experiment_configs[selected_experiment])
    print("Selected by validation macro-F1, then balanced accuracy:", selected_experiment)
    print("Selected validation macro-F1:", f"{selected_result['validation_macro_f1']:.4f}")
    print("Selected validation balanced accuracy:",
          f"{selected_result['validation_balanced_accuracy']:.4f}")
    print("Selected configuration:", selected_config)

    # Refit scaling on all of Session 1, then apply it unchanged to Session 2.
    final_scaler = StandardScaler()
    X_train_final_scaled = final_scaler.fit_transform(X_train)
    X_test_final_scaled = final_scaler.transform(X_test)
    if final_scaler.n_samples_seen_ != len(X_train):
        raise ValueError("Final scaler was not fitted on all Session 1 training records.")

    final_mlp_model = MLPClassifier(**selected_config, verbose=False)
    final_mlp_model.fit(X_train_final_scaled, y_train_encoded)
    if not np.array_equal(final_mlp_model.classes_, np.arange(len(label_encoder.classes_))):
        raise ValueError("Final model class indices do not match the participant label mapping.")

    # This is the only model prediction/evaluation on Session 2 in this experiment.
    y_test_final_predictions = final_mlp_model.predict(X_test_final_scaled)
    final_test_metrics = classification_metrics(y_test_encoded, y_test_final_predictions)
    print("Session 2 samples:", len(y_test_encoded))
    print("Session 2 accuracy:", f"{final_test_metrics['accuracy']:.4f}")
    print("Session 2 balanced accuracy:",
          f"{final_test_metrics['balanced_accuracy']:.4f}")
    print("Session 2 macro precision:", f"{final_test_metrics['macro_precision']:.4f}")
    print("Session 2 macro recall:", f"{final_test_metrics['macro_recall']:.4f}")
    print("Session 2 macro F1:", f"{final_test_metrics['macro_f1']:.4f}")

    final_result = {
        "selected_experiment": selected_experiment,
        "hidden_layer_sizes": str(selected_config["hidden_layer_sizes"]),
        "learning_rate_init": selected_config["learning_rate_init"],
        "alpha": selected_config["alpha"],
        "early_stopping": selected_config["early_stopping"],
        "random_state": selected_config["random_state"],
        "training_samples": len(X_train),
        "testing_samples": len(X_test),
        "validation_macro_f1": selected_result["validation_macro_f1"],
        "validation_balanced_accuracy": selected_result["validation_balanced_accuracy"],
        **{f"session_2_{name}": value for name, value in final_test_metrics.items()},
    }
    final_session2_results = pd.DataFrame([final_result])
    final_session2_results.to_csv(final_results_path, index=False)
    print("Saved final held-out evaluation to:", final_results_path)

Loaded the previously recorded one-time Session 2 evaluation; it was not rerun.


  selected_experiment hidden_layer_sizes  learning_rate_init  alpha  \
0   higher_alpha_0.01          (128, 64)              0.0005   0.01   

   early_stopping  random_state  training_samples  testing_samples  \
0            True            42              9876             9893   

   validation_macro_f1  validation_balanced_accuracy  session_2_accuracy  \
0             0.675472                      0.678416             0.42434   

   session_2_balanced_accuracy  session_2_macro_precision  \
0                     0.424335                    0.45664   

   session_2_macro_recall  session_2_macro_f1  
0                0.424335            0.415795  

In [16]:

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

print("Training participants:", len(np.unique(y_train_encoded)))
print("Testing participants:", len(np.unique(y_test_encoded)))

print("Training label range:",
      np.min(y_train_encoded), "to", np.max(y_train_encoded))

print("Testing label range:",
      np.min(y_test_encoded), "to", np.max(y_test_encoded))


Training samples: 9878
Testing samples: 9894
Training participants: 99
Testing participants: 99
Training label range: 0 to 98
Testing label range: 0 to 98


In [17]:
print("Session 1 rows:", (df_model["session"] == 1).sum())
print("Session 2 rows:", (df_model["session"] == 2).sum())

print("Rows in feature matrix:", len(X))
print("Rows in model dataframe:", len(df_model))

Session 1 rows: 9878
Session 2 rows: 9894
Rows in feature matrix: 19772
Rows in model dataframe: 19772


In [7]:
# Step 14: Repeated stratified validation on Session 1 only

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
)

CV_FOLDS = 5
CV_SEEDS = [42, 123, 2026]
CV_RESULTS_PATH = PROJECT_DIR / "results" / "metrics" / "mlp_repeated_session1_validation.csv"

if CV_RESULTS_PATH.exists():
    raise FileExistsError(
        f"Refusing to overwrite existing repeated-validation results: {CV_RESULTS_PATH}"
    )
if len(X_train) != 9876 or X_train.columns.tolist() != feature_columns:
    raise ValueError("Expected the corrected Session 1 training data and 46 ordered features.")

# Both configurations share all settings except alpha. Each seed controls both
# the outer stratified split and the estimator's stochastic initialization.
CV_BASE_CONFIG = {
    "hidden_layer_sizes": (128, 64),
    "activation": "relu",
    "solver": "adam",
    "batch_size": 256,
    "learning_rate_init": 0.0005,
    "max_iter": 150,
    "early_stopping": True,
    "validation_fraction": 0.15,
    "n_iter_no_change": 10,
    "tol": 0.0001,
    "verbose": False,
}
CV_CONFIGS = {
    "baseline_alpha_0.001": {**CV_BASE_CONFIG, "alpha": 0.001},
    "higher_alpha_0.01": {**CV_BASE_CONFIG, "alpha": 0.01},
}
CV_METRIC_NAMES = [
    "accuracy",
    "balanced_accuracy",
    "macro_precision",
    "macro_recall",
    "macro_f1",
]

fold_rows = []
for split_seed in CV_SEEDS:
    splitter = StratifiedKFold(
        n_splits=CV_FOLDS,
        shuffle=True,
        random_state=split_seed,
    )

    for fold_number, (fit_indices, validation_indices) in enumerate(
        splitter.split(X_train, y_train),
        start=1,
    ):
        X_fold_fit = X_train.iloc[fit_indices]
        X_fold_validation = X_train.iloc[validation_indices]
        y_fold_fit = y_train.iloc[fit_indices]
        y_fold_validation = y_train.iloc[validation_indices]

        fold_label_encoder = LabelEncoder()
        y_fold_fit_encoded = fold_label_encoder.fit_transform(y_fold_fit)
        if not y_fold_validation.isin(fold_label_encoder.classes_).all():
            raise ValueError(
                f"Validation fold has unseen participants (seed={split_seed}, fold={fold_number})."
            )
        y_fold_validation_encoded = fold_label_encoder.transform(y_fold_validation)

        # Fit all learned feature preprocessing on this fold's fit rows only.
        fold_scaler = StandardScaler()
        X_fold_fit_scaled = fold_scaler.fit_transform(X_fold_fit)
        X_fold_validation_scaled = fold_scaler.transform(X_fold_validation)
        if fold_scaler.n_samples_seen_ != len(fit_indices):
            raise ValueError("Fold scaler was not fitted exclusively on fold training rows.")

        for configuration_name, config in CV_CONFIGS.items():
            estimator_config = {**config, "random_state": split_seed}
            model = MLPClassifier(**estimator_config)
            model.fit(X_fold_fit_scaled, y_fold_fit_encoded)
            predictions = model.predict(X_fold_validation_scaled)
            precision, recall, f1, _ = precision_recall_fscore_support(
                y_fold_validation_encoded,
                predictions,
                average="macro",
                zero_division=0,
            )

            fold_rows.append({
                "row_type": "fold",
                "configuration": configuration_name,
                "alpha": config["alpha"],
                "split_seed": split_seed,
                "fold": fold_number,
                "fit_samples": len(fit_indices),
                "validation_samples": len(validation_indices),
                "iterations": model.n_iter_,
                "accuracy": accuracy_score(y_fold_validation_encoded, predictions),
                "balanced_accuracy": balanced_accuracy_score(
                    y_fold_validation_encoded, predictions
                ),
                "macro_precision": precision,
                "macro_recall": recall,
                "macro_f1": f1,
            })

fold_results = pd.DataFrame(fold_rows)
if len(fold_results) != len(CV_SEEDS) * CV_FOLDS * len(CV_CONFIGS):
    raise ValueError("Unexpected repeated-validation fold count.")

summary_rows = []
for configuration_name, group in fold_results.groupby("configuration", sort=False):
    summary = {
        "row_type": "summary",
        "configuration": configuration_name,
        "alpha": CV_CONFIGS[configuration_name]["alpha"],
        "split_seed": "42;123;2026",
        "fold": "mean +/- std across 15 folds",
        "fit_samples": int(group["fit_samples"].mean()),
        "validation_samples": int(group["validation_samples"].mean()),
        "iterations": group["iterations"].mean(),
    }
    for metric_name in CV_METRIC_NAMES:
        summary[metric_name] = group[metric_name].mean()
        summary[f"{metric_name}_std"] = group[metric_name].std(ddof=1)
    summary_rows.append(summary)

summary_results = pd.DataFrame(summary_rows)
repeated_validation_results = pd.concat(
    [fold_results, summary_results],
    ignore_index=True,
    sort=False,
)
repeated_validation_results.to_csv(CV_RESULTS_PATH, index=False)

print(f"Session 1 repeated validation: {CV_FOLDS} folds x {len(CV_SEEDS)} seeds")
print("Seeds:", CV_SEEDS)
print("Paired validation folds per configuration:", len(fold_results) // len(CV_CONFIGS))
print("Fold-level metrics:")
display(
    fold_results[
        [
            "configuration",
            "split_seed",
            "fold",
            "fit_samples",
            "validation_samples",
            "accuracy",
            "balanced_accuracy",
            "macro_precision",
            "macro_recall",
            "macro_f1",
        ]
    ]
)
print("Mean and standard deviation across folds:")
display(summary_results)
print("Saved new results to:", CV_RESULTS_PATH)

Session 1 repeated validation: 5 folds x 3 seeds
Seeds: [42, 123, 2026]
Paired validation folds per configuration: 15
Fold-level metrics:


,configuration,split_seed,fold,fit_samples,validation_samples,accuracy,balanced_accuracy,macro_precision,macro_recall,macro_f1
0,baseline_alpha_0.001,42,1,7900,1976,0.671053,0.670973,0.675200,0.670973,0.667567
1,higher_alpha_0.01,42,1,7900,1976,0.669028,0.668873,0.674284,0.668873,0.665940
2,baseline_alpha_0.001,42,2,7901,1975,0.688608,0.688650,0.694708,0.688650,0.687561
3,higher_alpha_0.01,42,2,7901,1975,0.686582,0.686603,0.694264,0.686603,0.686166
4,baseline_alpha_0.001,42,3,7901,1975,0.670886,0.671159,0.681031,0.671159,0.669434
5,higher_alpha_0.01,42,3,7901,1975,0.677468,0.677751,0.688596,0.677751,0.676259
6,baseline_alpha_0.001,42,4,7901,1975,0.685570,0.685354,0.693928,0.685354,0.682225
7,higher_alpha_0.01,42,4,7901,1975,0.692658,0.692477,0.700695,0.692477,0.690594
8,baseline_alpha_0.001,42,5,7901,1975,0.661772,0.661776,0.663517,0.661776,0.656620
9,higher_alpha_0.01,42,5,7901,1975,0.665823,0.665843,0.670619,0.665843,0.662275


Mean and standard deviation across folds:


,row_type,configuration,alpha,split_seed,fold,fit_samples,validation_samples,iterations,accuracy,accuracy_std,balanced_accuracy,balanced_accuracy_std,macro_precision,macro_precision_std,macro_recall,macro_recall_std,macro_f1,macro_f1_std
0,summary,baseline_alpha_0.001,0.001,42;123;2026,mean +/- std across 15 folds,7900,1975,113.0,0.668051,0.014464,0.668074,0.014406,0.674745,0.016053,0.668074,0.014406,0.665582,0.015185
1,summary,higher_alpha_0.01,0.010,42;123;2026,mean +/- std across 15 folds,7900,1975,116.4,0.670008,0.013951,0.670012,0.013881,0.675681,0.016433,0.670012,0.013881,0.667249,0.015204


Saved new results to: C:\Users\user\Desktop\DeepKey_DL_Project\results\metrics\mlp_repeated_session1_validation.csv
